In [1]:
# %%
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mne

mne.set_log_level("WARNING")

In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()

MANIFEST_PATH = PROJECT_ROOT / "data" / "derived" / "manifests" / "manifest_spontaneous_validated.csv"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EYES_KEEP = "closed"
FMIN, FMAX = 1.0, 40.0

BANDS_HZ = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
}

COLORS = {"awake": "steelblue", "ketamine": "firebrick"}

print("Figures will be saved to:", FIGURE_DIR)

Figures will be saved to: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures


In [3]:
# %%
# ============================================
# Section 1. Load manifest — eyes-closed only
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)
df = manifest[manifest["eyes"] == EYES_KEEP].copy()
df = df.sort_values(["subject_id", "recording_number"]).reset_index(drop=True)

print(f"Recordings: {len(df)}  |  Subjects: {df['subject_id'].nunique()}")
display(df.groupby("drug").size().rename("n_recordings"))

Recordings: 20  |  Subjects: 10


drug
awake       10
ketamine    10
Name: n_recordings, dtype: int64

In [4]:
# %%
# ============================================
# Section 2. Compute PSD for every recording
# ============================================
# psds[drug] -> list of arrays, one per recording, shape (n_channels, n_freqs)
# Each array is the mean across epochs within that recording.

psds   = {"awake": [], "ketamine": []}
freqs  = None
ch_info = None   # keep one mne.Info for topomaps

for _, row in df.iterrows():
    epochs = mne.io.read_epochs_eeglab(row["file_path"], verbose="ERROR")
    epochs.load_data()

    spectrum = epochs.compute_psd(
        method="welch",
        fmin=FMIN,
        fmax=FMAX,
        n_fft=int(epochs.info["sfreq"] * 2),   # 2-s windows
        n_overlap=int(epochs.info["sfreq"]),    # 50 % overlap
        verbose="ERROR",
    )

    # EpochsSpectrum.get_data() -> (n_epochs, n_channels, n_freqs)
    psd_mean = spectrum.get_data().mean(axis=0)   # -> (n_channels, n_freqs)

    if freqs is None:
        freqs = spectrum.freqs
    if ch_info is None:
        ch_info = epochs.info

    psds[row["drug"]].append(psd_mean)

# Stack to (n_recordings, n_channels, n_freqs)
for drug in psds:
    psds[drug] = np.array(psds[drug])

print("Awake shape:   ", psds["awake"].shape)
print("Ketamine shape:", psds["ketamine"].shape)
print("Frequencies:   ", freqs[0], "…", freqs[-1], "Hz  (", len(freqs), "bins)")

Awake shape:    (10, 62, 79)
Ketamine shape: (10, 62, 79)
Frequencies:    1.0 … 40.0 Hz  ( 79 bins)


In [12]:
# %%
# ============================================
# Section 3. Plot 1 — Grand-average PSD with SEM
# ============================================
# One curve per condition; average across channels first, then across subjects.
# SEM is computed across subjects (the meaningful unit of replication).

fig, ax = plt.subplots(figsize=(10, 5))

for drug, color in COLORS.items():
    per_subj = psds[drug].mean(axis=1)                      # (n_subj, n_freqs) — avg over channels
    grand    = per_subj.mean(axis=0)                        # (n_freqs,)
    sem      = per_subj.std(axis=0) / np.sqrt(len(per_subj))

    ax.semilogy(freqs, grand, label=drug.capitalize(), color=color, linewidth=2)
    ax.fill_between(
        freqs,
        np.maximum(grand - sem, 1e-30),
        grand + sem,
        alpha=0.25,
        color=color,
    )

# Shade canonical bands
band_colors = {"delta": "#e8e8e8", "theta": "#d8eaf7", "alpha": "#d4f0d4", "beta": "#fce8d4"}
for band, (flo, fhi) in BANDS_HZ.items():
    ax.axvspan(flo, fhi, alpha=0.15, color=band_colors[band], label=band)

ax.set_xlabel("Frequency (Hz)", fontsize=12)
ax.set_ylabel("Power spectral density (a.u.)", fontsize=12)
ax.legend(loc="upper right", fontsize=10)
ax.set_xlim(FMIN, FMAX)
fig.tight_layout()

out = FIGURE_DIR / "psd_grand_average.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/psd_grand_average.png


In [ ]:
# %%
# ============================================
# Section 4. Plot 2 — Per-subject PSD traces
# ============================================
# Thin lines = individual subjects; thick line = group mean.

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

subjects = sorted(df["subject_id"].unique())

for ax, drug in zip(axes, ["awake", "ketamine"]):
    color = COLORS[drug]
    per_subj = psds[drug].mean(axis=1)    # (n_subj, n_freqs)

    for i, sid in enumerate(subjects):
        ax.semilogy(freqs, per_subj[i], color=color, alpha=0.35, linewidth=1)

    grand = per_subj.mean(axis=0)
    ax.semilogy(freqs, grand, color=color, linewidth=2.5, label="Group mean")

    for band, (flo, fhi) in BANDS_HZ.items():
        ax.axvspan(flo, fhi, alpha=0.12, color=band_colors[band])
    ax.set_xlabel("Frequency (Hz)", fontsize=11)
    ax.set_xlim(FMIN, FMAX)
    ax.legend(fontsize=10)

axes[0].set_ylabel("Power spectral density (a.u.)", fontsize=11)
fig.tight_layout()

out = FIGURE_DIR / "psd_per_subject.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/psd_per_subject.png


In [7]:
# %%
# ============================================
# Section 5. Plot 3 — Ketamine / Awake power ratio
# ============================================
# Log-ratio per subject then averaged — highlights which frequencies
# change most under ketamine. Positive = more power under ketamine.

log_ratio_per_subj = np.log(psds["ketamine"].mean(axis=1)) - np.log(psds["awake"].mean(axis=1))
# (n_subj, n_freqs)

grand_ratio = log_ratio_per_subj.mean(axis=0)
sem_ratio   = log_ratio_per_subj.std(axis=0) / np.sqrt(log_ratio_per_subj.shape[0])

fig, ax = plt.subplots(figsize=(10, 4))

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.plot(freqs, grand_ratio, color="purple", linewidth=2)
ax.fill_between(
    freqs,
    grand_ratio - sem_ratio,
    grand_ratio + sem_ratio,
    alpha=0.25,
    color="purple",
)

for band, (flo, fhi) in BANDS_HZ.items():
    ax.axvspan(flo, fhi, alpha=0.12, color=band_colors[band])
    mid = (flo + fhi) / 2
    ax.text(mid, ax.get_ylim()[1] if ax.get_ylim()[1] < 10 else 2,
            band, ha="center", va="top", fontsize=9, color="grey")

ax.set_xlabel("Frequency (Hz)", fontsize=12)
ax.set_ylabel("Log power ratio\n(ketamine − awake)", fontsize=11)
ax.set_xlim(FMIN, FMAX)
fig.tight_layout()

out = FIGURE_DIR / "psd_log_ratio.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/psd_log_ratio.png


In [8]:
# %%
# ============================================
# Section 6. Plot 4 — Topographic PSD maps per band
# ============================================
# Grid: rows = bands, cols = [awake, ketamine, difference (ket-awake)]

bands = list(BANDS_HZ.keys())
n_bands = len(bands)

fig, axes = plt.subplots(n_bands, 3, figsize=(9, n_bands * 2.5))

for row_idx, (band, (flo, fhi)) in enumerate(BANDS_HZ.items()):
    freq_mask = (freqs >= flo) & (freqs < fhi)

    # (n_subj, n_channels) — mean power in band, averaged over subjects
    awake_topo = psds["awake"][:, :, freq_mask].mean(axis=(0, 2))
    ket_topo   = psds["ketamine"][:, :, freq_mask].mean(axis=(0, 2))
    diff_topo  = np.log(ket_topo) - np.log(awake_topo)

    # Absolute power maps (log scale for display)
    vmin_abs = np.log(min(awake_topo.min(), ket_topo.min()))
    vmax_abs = np.log(max(awake_topo.max(), ket_topo.max()))
    vlim_diff = np.abs(diff_topo).max()

    for col_idx, (topo, title, cmap, vmin, vmax) in enumerate([
        (np.log(awake_topo), "Awake",    "viridis", vmin_abs, vmax_abs),
        (np.log(ket_topo),   "Ketamine", "viridis", vmin_abs, vmax_abs),
        (diff_topo,          "Δ log power\n(ket − awake)", "RdBu_r", -vlim_diff, vlim_diff),
    ]):
        ax = axes[row_idx, col_idx]
        im, _ = mne.viz.plot_topomap(
            topo,
            ch_info,
            axes=ax,
            show=False,
            cmap=cmap,
            vlim=(vmin, vmax),
            contours=4,
        )
        if col_idx == 0:
            ax.set_ylabel(band, fontsize=11, labelpad=30, rotation=0, va="center")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()

out = FIGURE_DIR / "psd_topomap_per_band.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/psd_topomap_per_band.png


In [9]:
# %%
print("All figures saved to:", FIGURE_DIR)
for f in sorted(FIGURE_DIR.glob("psd_*.png")):
    print(" ", f.name)

All figures saved to: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures
  psd_grand_average.png
  psd_log_ratio.png
  psd_per_subject.png
  psd_topomap_per_band.png


In [10]:
# %%
# ============================================
# Section 7. Cluster-based permutation test on PSD
# Paired, within-subject test of awake vs. ketamine
# at every frequency in 1-40 Hz.
# ============================================

import numpy as np
from mne.stats import permutation_cluster_1samp_test

# Per-subject log power, averaged across channels.
# Shape: (n_subj, n_freqs) for each condition.
awake_lp    = np.log(psds["awake"].mean(axis=1))
ketamine_lp = np.log(psds["ketamine"].mean(axis=1))

# Within-subject difference (ketamine - awake) in log space
diff_lp = ketamine_lp - awake_lp        # (n_subj, n_freqs)

n_subj = diff_lp.shape[0]
print(f"Subjects: {n_subj}  |  Frequencies tested: {diff_lp.shape[1]}")

# Two-tailed paired test against zero. With n_subj = 10, the
# t-threshold for alpha=0.05 (df=9) is 2.262. permutation_cluster_1samp_test
# uses the sign-flip null, which is the correct permutation scheme for
# a paired design with a small N.

T_OBS, clusters, cluster_pv, H0 = permutation_cluster_1samp_test(
    diff_lp,
    n_permutations=10_000,
    threshold=2.262,        # df = n_subj - 1 = 9; t-crit at p=0.05 two-tailed
    tail=0,                 # two-sided
    out_type="mask",
    seed=0,
    verbose=False,
)

# Pretty-print clusters
print(f"\n{len(clusters)} cluster(s) found:")
for ci, (cmask, p) in enumerate(zip(clusters, cluster_pv)):
    fmin, fmax = freqs[cmask].min(), freqs[cmask].max()
    direction = "ketamine > awake" if T_OBS[cmask].mean() > 0 else "ketamine < awake"
    sig = "*" if p < 0.05 else ""
    print(f"  cluster {ci+1}: {fmin:5.2f}-{fmax:5.2f} Hz   "
          f"p = {p:.4f}{sig}   direction: {direction}")

Subjects: 10  |  Frequencies tested: 79

4 cluster(s) found:
  cluster 1: 37.50-38.00 Hz   p = 0.3340   direction: ketamine > awake
  cluster 2:  1.50- 9.00 Hz   p = 0.0137*   direction: ketamine < awake
  cluster 3: 11.00-12.50 Hz   p = 0.1621   direction: ketamine < awake
  cluster 4: 13.50-24.50 Hz   p = 0.0039*   direction: ketamine < awake


In [17]:
# %%
# ============================================
# Section 8. Log-ratio figure with cluster significance overlay
# Re-renders Figure 4 with horizontal bars marking significant clusters.
# ============================================

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

grand_ratio = diff_lp.mean(axis=0)
sem_ratio   = diff_lp.std(axis=0) / np.sqrt(n_subj)
ax.plot(freqs, grand_ratio, color="purple", linewidth=2)
ax.fill_between(freqs,
                grand_ratio - sem_ratio,
                grand_ratio + sem_ratio,
                alpha=0.25, color="purple")

# Band shading
for band, (flo, fhi) in BANDS_HZ.items():
    ax.axvspan(flo, fhi, alpha=0.12, color=band_colors[band])

# Significant clusters: draw a horizontal bar near the top of the plot
ylim = ax.get_ylim()
bar_y = ylim[1] - 0.05 * (ylim[1] - ylim[0])
for cmask, p in zip(clusters, cluster_pv):
    if p < 0.05:
        # Handle both possible return types from MNE
        if isinstance(cmask, tuple):
            # tuple of slice objects, e.g. (slice(3, 19, None),)
            idx = np.arange(*cmask[0].indices(len(freqs)))
        else:
            # boolean mask
            idx = np.where(cmask)[0]

        ax.plot(freqs[idx],
                np.full(len(idx), bar_y),
                color="black", linewidth=4, solid_capstyle="butt")
ax.set_xlabel("Frequency (Hz)", fontsize=12)
ax.set_ylabel("Log power ratio\n(ketamine − awake)", fontsize=11)
ax.set_xlim(FMIN, FMAX)
fig.tight_layout()

out = FIGURE_DIR / "psd_log_ratio_with_clusters.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/psd_log_ratio_with_clusters.png


In [15]:
# %%
# ============================================
# Section 9. Per-band Wilcoxon signed-rank tests
# Each subject contributes one log-power difference per band.
# ============================================

from scipy import stats

# Average each subject's PSD across channels first, then average
# across the frequency bins inside each band, then take log.
band_rows = []
for band, (flo, fhi) in BANDS_HZ.items():
    fmask = (freqs >= flo) & (freqs < fhi)

    awake_band    = np.log(psds["awake"][:,    :, fmask].mean(axis=(1, 2)))
    ketamine_band = np.log(psds["ketamine"][:, :, fmask].mean(axis=(1, 2)))
    diff = ketamine_band - awake_band

    w_stat, w_p = stats.wilcoxon(diff, alternative="two-sided",
                                 zero_method="wilcox", mode="exact")

    band_rows.append({
        "band":            band,
        "freq_range_Hz":   f"{flo}-{fhi}",
        "mean_log_diff":   float(diff.mean()),
        "median_log_diff": float(np.median(diff)),
        "n_pos":           int((diff > 0).sum()),
        "n_neg":           int((diff < 0).sum()),
        "W":               float(w_stat),
        "p":               float(w_p),
    })

band_df = pd.DataFrame(band_rows)

# Bonferroni-Holm correction over 4 bands
from statsmodels.stats.multitest import multipletests
reject, p_adj, _, _ = multipletests(band_df["p"].values, method="holm")
band_df["p_holm"] = p_adj
band_df["significant"] = reject

print("Per-band paired Wilcoxon tests (ketamine − awake, log power):")
print(band_df.round(4).to_string(index=False))

band_df.to_csv(FIGURE_DIR.parent / "psd_band_wilcoxon.csv", index=False)
print("\nSaved per-band table to psd_band_wilcoxon.csv")

Per-band paired Wilcoxon tests (ketamine − awake, log power):
 band freq_range_Hz  mean_log_diff  median_log_diff  n_pos  n_neg   W      p  p_holm  significant
delta       1.0-4.0        -0.3701          -0.4285      2      8 5.0 0.0195  0.0391         True
theta       4.0-8.0        -0.7061          -0.6974      0     10 0.0 0.0020  0.0078         True
alpha      8.0-13.0        -0.6373          -0.8148      1      9 7.0 0.0371  0.0391         True
 beta     13.0-30.0        -0.4849          -0.4206      0     10 0.0 0.0020  0.0078         True

Saved per-band table to psd_band_wilcoxon.csv
